# Prerequisites - Hierarchical IDS (Fog-Cloud)

Setup notebook for the hierarchical IDS project. Run the cells top to bottom once;
the final cell prints a readiness checklist.

Covers:
1. Installing missing packages
2. Setting up EvoloPy-FS (evolutionary feature selection)
3. Core imports + version check
4. Project path wiring (so `src.*` imports work from this folder)
5. Dataset availability + load sanity check


---
## 1. Install missing packages

Already present from `requirements.txt`: pandas, numpy, scikit-learn, matplotlib,
seaborn, imbalanced-learn, xgboost, joblib.

The plan adds LightGBM and CatBoost to the model suite - those aren't installed yet.


In [ ]:
import sys, subprocess
import importlib.metadata as meta

REQUIRED = [
    'pandas', 'numpy', 'scikit-learn', 'matplotlib', 'seaborn',
    'imbalanced-learn', 'xgboost', 'joblib', 'scipy', 'tqdm',
    'lightgbm', 'catboost',
]

missing = []
for pkg in REQUIRED:
    try:
        meta.version(pkg)
    except meta.PackageNotFoundError:
        missing.append(pkg)

if missing:
    print('Installing:', ', '.join(missing))
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
    print('Done.')
else:
    print('All packages already installed.')


In [ ]:
import os, subprocess
from pathlib import Path

EVOLOPY_URL = 'https://github.com/7ossam81/EvoloPy.git'
EXTERNAL_DIR = (Path.cwd().parent if Path.cwd().name == 'kar' else Path.cwd()) / 'external'
EVOLOPY_DIR = EXTERNAL_DIR / 'EvoloPy'

# presence is judged by the package dir, not the folder - a half-finished clone
# leaves the folder behind and would otherwise look like a success
if (EVOLOPY_DIR / 'EvoloPy' / 'optimizers').is_dir():
    print(f'EvoloPy already present at {EVOLOPY_DIR.resolve()}')
else:
    EXTERNAL_DIR.mkdir(exist_ok=True)
    print(f'Cloning EvoloPy into {EVOLOPY_DIR} ...')
    subprocess.check_call(['git', 'clone', '--depth', '1', EVOLOPY_URL, str(EVOLOPY_DIR)])
    print('Done.')

In [ ]:
import os, subprocess
from pathlib import Path

EVOLOPY_URL = 'https://github.com/RaneemQaddoura/EvoloPy-FS.git'
EXTERNAL_DIR = Path('..') / 'external'
EVOLOPY_DIR = EXTERNAL_DIR / 'EvoloPy-FS'

if EVOLOPY_DIR.exists():
    print(f'EvoloPy-FS already present at {EVOLOPY_DIR.resolve()}')
else:
    EXTERNAL_DIR.mkdir(exist_ok=True)
    print(f'Cloning EvoloPy-FS into {EVOLOPY_DIR} ...')
    subprocess.check_call(['git', 'clone', '--depth', '1', EVOLOPY_URL, str(EVOLOPY_DIR)])
    print('Done.')


---
## 3. Core imports

Everything the pipeline needs, in one place. Run this cell at the top of any
working notebook.


In [ ]:
# --- stdlib ---
import os
import sys
import time
import json
import warnings
from pathlib import Path

# --- data ---
import numpy as np
import pandas as pd

# --- viz ---
import matplotlib.pyplot as plt
import seaborn as sns

# --- preprocessing ---
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score

# --- imbalance handling ---
from imblearn.over_sampling import SMOTE, BorderlineSMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler

# --- models ---
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# --- metrics ---
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    auc,
)

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

print('Imports OK')


In [ ]:
# Version check
import importlib.metadata as meta

print('=' * 55)
print('ENVIRONMENT')
print('=' * 55)
print('  {:<20} {}'.format('python', sys.version.split()[0]))
for pkg in ['pandas', 'numpy', 'scikit-learn', 'imbalanced-learn',
            'xgboost', 'lightgbm', 'catboost', 'matplotlib', 'seaborn']:
    try:
        print('  {:<20} {}'.format(pkg, meta.version(pkg)))
    except meta.PackageNotFoundError:
        print('  {:<20} {}'.format(pkg, 'MISSING'))


---
## 4. Project paths

This notebook lives in `kar/`, one level below the repo root. Adding the root to
`sys.path` lets `from src.config import ...` resolve the same way it does for
`src/main.py`.


In [ ]:
REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'kar' else Path.cwd()
REPO_ROOT = REPO_ROOT.resolve()

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.config import (
    COLUMN_NAMES,
    CATEGORICAL_COLS,
    LABEL_COL,
    ATTACK_CATEGORY_MAP,
    CATEGORIES,
    RANDOM_STATE,
    DATA_DIR,
    MODELS_DIR,
    RESULTS_DIR,
    TRAIN_FILE,
    TEST_FILE,
)

np.random.seed(RANDOM_STATE)

print(f'  repo root : {REPO_ROOT}')
print(f'  data      : {DATA_DIR}')
print(f'  models    : {MODELS_DIR}')
print(f'  results   : {RESULTS_DIR}')
print(f'  seed      : {RANDOM_STATE}')


---
## 5. Dataset check

NSL-KDD ships as headerless comma-separated `.txt`. `COLUMN_NAMES` supplies the
43 headers (41 features + `intrusion_type` + `difficulty`).


In [ ]:
for path in [TRAIN_FILE, TEST_FILE]:
    exists = os.path.exists(path)
    size = f'{os.path.getsize(path) / 1e6:.1f} MB' if exists else '-'
    status = 'OK' if exists else 'MISSING'
    print(f'  [{status:>7}] {os.path.basename(path):<16} {size}')


In [ ]:
train_df = pd.read_csv(TRAIN_FILE, names=COLUMN_NAMES)
test_df = pd.read_csv(TEST_FILE, names=COLUMN_NAMES)

train_df = train_df.drop('difficulty', axis=1)
test_df = test_df.drop('difficulty', axis=1)

print(f'  train : {train_df.shape}')
print(f'  test  : {test_df.shape}')
print(f'  nulls : {train_df.isnull().sum().sum()} train / {test_df.isnull().sum().sum()} test')

train_df.head()


In [ ]:
# Attack category distribution - the imbalance the plan calls out
cat = train_df[LABEL_COL].map(ATTACK_CATEGORY_MAP)

print('  --- Train: 5-category distribution ---')
for name in CATEGORIES:
    n = int((cat == name).sum())
    print(f'    {name:<8} {n:>7}  ({n / len(train_df) * 100:5.2f}%)')

unmapped = train_df.loc[cat.isna(), LABEL_COL].unique()
if len(unmapped):
    print(f'\n  WARNING - labels missing from ATTACK_CATEGORY_MAP: {list(unmapped)}')


---
## 6. Readiness check


In [ ]:
checks = []

for pkg in ['pandas', 'numpy', 'scikit-learn', 'imbalanced-learn',
            'xgboost', 'lightgbm', 'catboost']:
    try:
        meta.version(pkg)
        checks.append(('package: ' + pkg, True))
    except meta.PackageNotFoundError:
        checks.append(('package: ' + pkg, False))

checks.append(('EvoloPy cloned', (EVOLOPY_DIR / 'EvoloPy' / 'optimizers').is_dir()))
checks.append(('train data', os.path.exists(TRAIN_FILE)))
checks.append(('test data', os.path.exists(TEST_FILE)))
checks.append(('src.config importable', 'COLUMN_NAMES' in dir()))
checks.append(('models/ dir', os.path.isdir(MODELS_DIR)))
checks.append(('results/ dir', os.path.isdir(RESULTS_DIR)))

print('=' * 55)
print('READINESS')
print('=' * 55)
for label, ok in checks:
    print('  [{:>4}] {}'.format('OK' if ok else 'FAIL', label))

n_fail = sum(1 for _, ok in checks if not ok)
print('=' * 55)
print('  All prerequisites met.' if n_fail == 0 else f'  {n_fail} item(s) need attention.')


---
---
# STEP 1 - Data Preprocessing & Balancing

Implements section 1.1 of the implementation plan.

| Sub-step | What |
|---|---|
| 1.1 | Ingestion - load raw NSL-KDD |
| 1.2 | Cleaning - nulls, infinities, duplicates, constant columns |
| 1.3 | Encoding - categorical -> numeric |
| 1.4 | Labels - binary (Level 1) + multiclass (Level 2) |
| 1.5 | Scaling - Min-Max to [0, 1] |
| 1.6 | Balancing Level 1 - Normal vs Attack |
| 1.7 | Balancing Level 2 - attack-type categories |
| 1.8 | Save artifacts |

**Hierarchical split:** Level 1 sees *all* traffic and answers Normal-vs-Attack.
Level 2 only ever sees rows Level 1 flagged as attacks, so it is trained on
**attack rows only** - `Normal` is excluded from its label set.


## 1.1 Ingestion


In [ ]:
train_raw = pd.read_csv(TRAIN_FILE, names=COLUMN_NAMES)
test_raw = pd.read_csv(TEST_FILE, names=COLUMN_NAMES)

# 'difficulty' is an NSL-KDD annotation, not a feature
train_raw = train_raw.drop('difficulty', axis=1)
test_raw = test_raw.drop('difficulty', axis=1)

print(f'  train : {train_raw.shape}')
print(f'  test  : {test_raw.shape}')


## 1.2 Cleaning

NSL-KDD is already de-duplicated and complete, so these checks mostly confirm a
clean slate. They stay in because the plan targets other datasets (IoT-23,
Edge-IIoTset) where they will actually fire.

`num_outbound_cmds` is all-zero in NSL-KDD - a constant column carries no signal
and is dropped.


In [ ]:
def clean(df, name, drop_cols=None):
    """Impute nulls, replace infinities, drop duplicates and constant columns."""
    df = df.copy()
    n_before = len(df)

    num_cols = df.select_dtypes(include=np.number).columns

    # infinities -> NaN, so a single imputation pass handles both
    n_inf = int(np.isinf(df[num_cols].to_numpy()).sum())
    df[num_cols] = df[num_cols].replace([np.inf, -np.inf], np.nan)

    # median imputation for numeric columns
    n_null = int(df.isnull().sum().sum())
    if n_null:
        df[num_cols] = df[num_cols].fillna(df[num_cols].median())

    df = df.drop_duplicates().reset_index(drop=True)
    n_dupes = n_before - len(df)

    # constant columns (decided on train, applied to both)
    if drop_cols is None:
        drop_cols = [c for c in num_cols if df[c].nunique() <= 1]
    df = df.drop(columns=[c for c in drop_cols if c in df.columns])

    print(f'  {name}:')
    print(f'    infinities replaced : {n_inf}')
    print(f'    nulls imputed       : {n_null}')
    print(f'    duplicates dropped  : {n_dupes}')
    print(f'    constant cols       : {drop_cols if drop_cols else "none"}')
    print(f'    shape               : {df.shape}')
    return df, drop_cols


train_df, const_cols = clean(train_raw, 'train')
test_df, _ = clean(test_raw, 'test', drop_cols=const_cols)


## 1.3 Encoding

`protocol_type`, `service`, `flag` are strings. Label-encoded rather than one-hot
because the model suite is entirely tree-based (RF/XGB/LGBM/CatBoost), which splits
on ordinal codes without assuming an ordering - and one-hot on `service` alone would
add ~70 sparse columns for the feature-selection search to wade through.

The encoder is fit on **train + test combined** so a category appearing only in test
doesn't raise at transform time.


In [ ]:
encoders = {}

for col in CATEGORICAL_COLS:
    le = LabelEncoder()
    le.fit(pd.concat([train_df[col], test_df[col]]).unique())
    train_df[col] = le.transform(train_df[col])
    test_df[col] = le.transform(test_df[col])
    encoders[col] = le
    print(f'  {col:<15} {len(le.classes_):>3} categories')


## 1.4 Labels

Two targets built from the same `intrusion_type` column:

- **Level 1 (binary)** - `0` = normal, `1` = attack. Uses every row.
- **Level 2 (multiclass)** - DoS / Probe / R2L / U2R. Uses **attack rows only**.

The test set contains attack types absent from train (that's the point of NSL-KDD's
test split) - the category map folds them into the same 4 buckets, so the label
space stays consistent.


In [ ]:
FEATURES = [c for c in train_df.columns if c != LABEL_COL]

# ---------- Level 1: binary, all rows ----------
X_train_l1 = train_df[FEATURES].copy()
X_test_l1 = test_df[FEATURES].copy()
y_train_l1 = (train_df[LABEL_COL] != 'normal').astype(int).values
y_test_l1 = (test_df[LABEL_COL] != 'normal').astype(int).values

# ---------- Level 2: multiclass, attack rows only ----------
train_cat = train_df[LABEL_COL].map(ATTACK_CATEGORY_MAP)
test_cat = test_df[LABEL_COL].map(ATTACK_CATEGORY_MAP)

unmapped = set(train_df.loc[train_cat.isna(), LABEL_COL]) | set(test_df.loc[test_cat.isna(), LABEL_COL])
assert not unmapped, f'labels missing from ATTACK_CATEGORY_MAP: {unmapped}'

train_atk = train_cat != 'Normal'
test_atk = test_cat != 'Normal'

X_train_l2 = train_df.loc[train_atk, FEATURES].copy()
X_test_l2 = test_df.loc[test_atk, FEATURES].copy()
y_train_l2 = train_cat[train_atk].values
y_test_l2 = test_cat[test_atk].values

print(f'  features : {len(FEATURES)}')
print(f'  L1 train : {X_train_l1.shape}  normal={np.sum(y_train_l1 == 0)}  attack={np.sum(y_train_l1 == 1)}')
print(f'  L1 test  : {X_test_l1.shape}  normal={np.sum(y_test_l1 == 0)}  attack={np.sum(y_test_l1 == 1)}')
print(f'  L2 train : {X_train_l2.shape}')
print(f'  L2 test  : {X_test_l2.shape}')


In [ ]:
# Level 2 imbalance - this is the hard part of the whole project
vc = pd.Series(y_train_l2).value_counts()
print('  --- L2 train distribution ---')
for name, n in vc.items():
    print(f'    {name:<8} {n:>7}  ({n / len(y_train_l2) * 100:6.2f}%)')
print(f'\n  imbalance ratio: {vc.max() / vc.min():.0f}:1')


## 1.5 Scaling

Min-Max to `[0, 1]`. Chosen over StandardScaler because the metaheuristic feature
selectors in Step 2 operate on a bounded search space, and distance-sensitive fitness
evaluators behave better on a common range.

Fit on **train only** - fitting on test would leak test distribution into training.


In [ ]:
scalers = {}

def scale(X_tr, X_te, key):
    sc = MinMaxScaler()
    X_tr_s = pd.DataFrame(sc.fit_transform(X_tr), columns=X_tr.columns, index=X_tr.index)
    X_te_s = pd.DataFrame(sc.transform(X_te), columns=X_te.columns, index=X_te.index)
    # test values outside the train range land outside [0,1]; clip them back
    X_te_s = X_te_s.clip(0, 1)
    scalers[key] = sc
    print(f'  {key}: scaled {X_tr_s.shape[1]} features')
    return X_tr_s, X_te_s


X_train_l1, X_test_l1 = scale(X_train_l1, X_test_l1, 'level1')
X_train_l2, X_test_l2 = scale(X_train_l2, X_test_l2, 'level2')


## 1.6 Balancing - Level 1 (binary)

> Normal vs Attack in NSL-KDD is already close to even (~53/46), so this is a light
> touch, not a rescue. Plain SMOTE is enough. The cell is written to be a no-op when
> the classes are already within tolerance, so it stays honest on other datasets too.


In [ ]:
def show_dist(y, title):
    vc = pd.Series(y).value_counts().sort_index()
    print(f'  {title}')
    for k, n in vc.items():
        print(f'    {str(k):<8} {n:>7}  ({n / len(y) * 100:6.2f}%)')
    print(f'    {"total":<8} {len(y):>7}')


show_dist(y_train_l1, 'before:')

vc = pd.Series(y_train_l1).value_counts()
ratio = vc.max() / vc.min()

if ratio < 1.5:
    print(f'\n  ratio {ratio:.2f}:1 - already balanced, SMOTE skipped')
    X_train_l1_bal, y_train_l1_bal = X_train_l1, y_train_l1
else:
    sm = SMOTE(random_state=RANDOM_STATE)
    X_train_l1_bal, y_train_l1_bal = sm.fit_resample(X_train_l1, y_train_l1)
    print()
    show_dist(y_train_l1_bal, 'after SMOTE:')


## 1.7 Balancing - Level 2 (multiclass)

The real problem: **U2R has 52 training rows against DoS's 45,927** - an 883:1 ratio.

Two guards matter here:

1. **`k_neighbors` must be smaller than the rarest class.** SMOTE interpolates between
   a sample and its *k* nearest same-class neighbours; the default `k=5` needs at least
   6 members. It survives U2R's 52, but hard-coding 5 breaks the moment a rarer class
   shows up - so it's derived from the data.
2. **Don't oversample everything to DoS's 45,927.** That would synthesise ~45,000 fake
   U2R rows from 52 real ones - a ratio of roughly 880 fabricated samples per genuine
   one, which manufactures a decision boundary rather than learning it. Capping the
   target keeps the minority classes represented without drowning the real signal.

`SAMPLING_STRATEGY` below is the knob: `'cap'` lifts rare classes to a ceiling,
`'full'` equalises everything (the naive version, kept for comparison).


In [ ]:
SAMPLING_STRATEGY = 'cap'   # 'cap' | 'full'
CAP_MULTIPLIER = 20         # rare classes lifted to at most 20x their original size
CAP_FLOOR = 1000            # ...but always to at least this many

show_dist(y_train_l2, 'before:')

counts = pd.Series(y_train_l2).value_counts()
n_min = counts.min()

# guard 1: k_neighbors must be < the rarest class size
k = min(5, int(n_min) - 1)
if k < 1:
    raise ValueError(f'rarest class has {n_min} sample(s) - too few for SMOTE')

# guard 2: build the target distribution
if SAMPLING_STRATEGY == 'full':
    strategy = 'auto'
else:
    target = {}
    for cls, n in counts.items():
        goal = max(CAP_FLOOR, n * CAP_MULTIPLIER)
        target[cls] = int(min(counts.max(), max(n, goal)))
    strategy = target

print(f'\n  k_neighbors : {k}  (rarest class = {n_min})')
print(f'  strategy    : {SAMPLING_STRATEGY}')
if isinstance(strategy, dict):
    for cls, n in strategy.items():
        print(f'    {cls:<8} {counts[cls]:>7} -> {n:>7}')


In [ ]:
sm = SMOTE(random_state=RANDOM_STATE, k_neighbors=k, sampling_strategy=strategy)
X_train_l2_bal, y_train_l2_bal = sm.fit_resample(X_train_l2, y_train_l2)

show_dist(y_train_l2_bal, 'after SMOTE:')

synth = len(y_train_l2_bal) - len(y_train_l2)
print(f'\n  synthetic rows created: {synth:,}')


### Alternative samplers

The plan also lists Borderline-SMOTE and ADASYN for Level 2. Both concentrate
synthetic points near the decision boundary rather than spreading them uniformly,
which usually helps when classes overlap.

ADASYN can fail outright on very small classes (it needs each minority sample to have
neighbours of a *different* class), so it is wrapped in a try/except rather than
assumed to work. Run this to compare - the winner gets promoted in Step 3.


In [ ]:
VARIANTS = {
    'SMOTE': SMOTE(random_state=RANDOM_STATE, k_neighbors=k, sampling_strategy=strategy),
    'BorderlineSMOTE': BorderlineSMOTE(random_state=RANDOM_STATE, k_neighbors=k, sampling_strategy=strategy),
    'ADASYN': ADASYN(random_state=RANDOM_STATE, n_neighbors=k, sampling_strategy=strategy),
}

resampled = {}
for name, sampler in VARIANTS.items():
    try:
        Xr, yr = sampler.fit_resample(X_train_l2, y_train_l2)
        resampled[name] = (Xr, yr)
        vc = pd.Series(yr).value_counts().sort_index()
        summary = '  '.join(f'{c}={n}' for c, n in vc.items())
        print(f'  [  OK  ] {name:<16} {len(yr):>7} rows | {summary}')
    except Exception as e:
        print(f'  [ FAIL ] {name:<16} {type(e).__name__}: {e}')


## 1.8 Save artifacts

Written to `data/processed/` so Step 2 (feature selection) can load straight in
without re-running any of the above. Encoders and scalers are saved alongside -
inference needs the exact same transforms, and refitting them later would silently
produce different numbers.


In [ ]:
import joblib

PROCESSED_DIR = REPO_ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

np.savez_compressed(
    PROCESSED_DIR / 'level1_binary.npz',
    X_train=X_train_l1_bal.to_numpy(dtype=np.float32) if hasattr(X_train_l1_bal, 'to_numpy') else np.asarray(X_train_l1_bal, dtype=np.float32),
    y_train=np.asarray(y_train_l1_bal),
    X_test=X_test_l1.to_numpy(dtype=np.float32),
    y_test=np.asarray(y_test_l1),
    features=np.array(FEATURES, dtype=object),
)

np.savez_compressed(
    PROCESSED_DIR / 'level2_multiclass.npz',
    X_train=np.asarray(X_train_l2_bal, dtype=np.float32),
    y_train=np.asarray(y_train_l2_bal, dtype=object),
    X_test=X_test_l2.to_numpy(dtype=np.float32),
    y_test=np.asarray(y_test_l2, dtype=object),
    features=np.array(FEATURES, dtype=object),
)

joblib.dump({'encoders': encoders, 'scalers': scalers, 'features': FEATURES,
             'const_cols_dropped': const_cols},
            PROCESSED_DIR / 'transformers.pkl')

for f in sorted(PROCESSED_DIR.iterdir()):
    print(f'  {f.name:<28} {f.stat().st_size / 1e6:>7.2f} MB')


---
## Step 1 complete

Ready for Step 2 (evolutionary feature selection):

| Artifact | Contents |
|---|---|
| `level1_binary.npz` | Normal-vs-Attack, all rows, balanced |
| `level2_multiclass.npz` | DoS/Probe/R2L/U2R, attack rows only, balanced |
| `transformers.pkl` | Fitted encoders + scalers + feature list |

**Carry forward into Step 2:** the feature search must be fit on the *training* split
only. Selecting features while the test set is visible leaks it, and the 108-experiment
table in section 1.4 would report scores that don't survive contact with real traffic.


---
---
# STEP 2 - Evolutionary Feature Selection

Implements section 1.2 of the implementation plan.

| Sub-step | What |
|---|---|
| 2.1 | Wiring in the metaheuristics |
| 2.2 | Loading Step 1's artifacts |
| 2.3 | The wrapper fitness function |
| 2.4 | Running the search - 6 optimizers x 2 levels |
| 2.5 | Results - convergence and feature frequency |
| 2.6 | Choosing the subsets to carry into Step 3 |

The plan asks for 6 metaheuristics producing 3 feature subsets for Level 1 and 3 for
Level 2. All 6 are run against **both** levels - 12 searches - and the top 3 per level
by search fitness are promoted. Running every algorithm on both levels costs a few
minutes and makes the promotion evidence-based rather than assigned up front.

---
## 2.1 Wiring in the metaheuristics

> **The plan names EvoloPy-FS, but that repository is empty.**
> `RaneemQaddoura/EvoloPy-FS` contains a README and nothing else - no source on any
> branch. Its own README points at the upstream **EvoloPy** toolbox, which is where the
> optimizers actually live. So: clone EvoloPy, and write the feature-selection wrapper
> ourselves in 2.3. That wrapper is the only thing EvoloPy-FS would have added, and
> writing it here means the fitness function is visible and tunable rather than buried
> in a dependency.

**One substitution.** The plan lists GA, PSO, GWO, **ALO**, WOA, MFO. EvoloPy ships
BAT, CS, DE, FFA, GA, GWO, HHO, JAYA, MFO, MVO, PSO, SCA, SSA and WOA - no Ant Lion
Optimizer. **MVO** (Multi-Verse Optimizer) takes ALO's slot: it is named in the
EvoloPy-FS README's own optimizer list, which makes it the closest in-family
replacement.

In [ ]:
import sys, subprocess
from pathlib import Path

EVOLOPY_DIR = REPO_ROOT / 'external' / 'EvoloPy'

if not (EVOLOPY_DIR / 'EvoloPy' / 'optimizers').is_dir():
    EVOLOPY_DIR.parent.mkdir(exist_ok=True)
    print(f'Cloning EvoloPy into {EVOLOPY_DIR} ...')
    subprocess.check_call(['git', 'clone', '--depth', '1',
                           'https://github.com/7ossam81/EvoloPy.git', str(EVOLOPY_DIR)])

if str(EVOLOPY_DIR) not in sys.path:
    sys.path.insert(0, str(EVOLOPY_DIR))

from EvoloPy.optimizers import GA, PSO, GWO, WOA, MFO, MVO

# name -> callable, all sharing the signature (objf, lb, ub, dim, popsize, iters)
OPTIMIZERS = {
    'GA':  GA.GA,     # Genetic Algorithm
    'PSO': PSO.PSO,   # Particle Swarm Optimization
    'GWO': GWO.GWO,   # Grey Wolf Optimizer
    'WOA': WOA.WOA,   # Whale Optimization Algorithm
    'MFO': MFO.MFO,   # Moth-Flame Optimization
    'MVO': MVO.MVO,   # Multi-Verse Optimizer (stands in for ALO)
}

print(f'  EvoloPy at {EVOLOPY_DIR}')
print(f'  optimizers: {", ".join(OPTIMIZERS)}')

---
## 2.2 Loading Step 1's artifacts

Step 2 reads `data/processed/` rather than re-deriving anything, so it runs standalone
after a kernel restart. The two levels differ in more than their labels - Level 2 has
four classes and a residual imbalance even after capped SMOTE - so each carries the
averaging mode its F1 should use.

In [ ]:
PROCESSED_DIR = REPO_ROOT / 'data' / 'processed'

LEVELS = {}
for key, fname, avg in [('level1', 'level1_binary.npz', 'binary'),
                        ('level2', 'level2_multiclass.npz', 'macro')]:
    d = np.load(PROCESSED_DIR / fname, allow_pickle=True)
    LEVELS[key] = dict(
        X_train=d['X_train'], y_train=d['y_train'],
        X_test=d['X_test'], y_test=d['y_test'],
        features=list(d['features']), average=avg,
    )
    print(f'  {key:<7} train {d["X_train"].shape}  test {d["X_test"].shape}  ({avg} F1)')

FEATURES = LEVELS['level1']['features']
DIM = len(FEATURES)
print(f'\n  search space: {DIM} features -> 2^{DIM} = {2**DIM:.3e} possible subsets')

---
## 2.3 The wrapper fitness function

This is the piece EvoloPy-FS would have supplied. Three decisions define it.

**1. Continuous positions, binary subsets.** Every optimizer here searches a continuous
box - each agent is a point in `[0, 1]^40`. A feature subset is binary, so the position
is thresholded at `0.5`: dimension *i* above the threshold means "keep feature *i*". The
optimizers still move through a smooth space, which is what their update rules assume;
only the evaluation is discrete.

**2. Fitness balances error against subset size.** The standard wrapper form:

$$\text{fitness} = \alpha \cdot (1 - F_1) + \beta \cdot \frac{k}{n}$$

with $\alpha = 0.99$, $\beta = 0.01$, $k$ selected out of $n = 40$. Minimised, so lower
is better. The lopsided weights are deliberate - detection quality dominates, and the
size term only breaks ties between subsets that classify equally well. That
tie-breaking is what matters for the edge deployment in Part 2: two subsets scoring the
same F1 are not equal if one reads 12 features off the wire and the other reads 31.

**3. Evaluated on a subsample, with an inner validation split.** A search is
`popsize x iters` model fits - a thousand per optimizer. Fitting each on 125k rows would
put this in hours. A stratified 15k-row subsample with a 70/30 inner split keeps a single
evaluation near 30ms and the whole search under a minute, and it is a *relative* ranking
that matters here, not an absolute score.

> **The inner split comes out of the training data only.** The NSL-KDD test set is not
> touched anywhere in this section. Selecting features while the test set is visible is
> the classic way to publish a number that evaporates on real traffic - the point Step 1
> closed on.

The empty subset is the one degenerate case: a position below `0.5` in all 40 dimensions
selects nothing and cannot be fitted at all. It returns the worst possible fitness of
`1.0` so the search moves away from it.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score

ALPHA, BETA = 0.99, 0.01   # error weight, subset-size weight
FIT_SAMPLE = 15000         # rows used per fitness evaluation
FIT_DEPTH = 12             # depth cap on the wrapper's Decision Tree


def make_fitness(X, y, average, n_sample=FIT_SAMPLE, seed=RANDOM_STATE):
    """Build an objective function over feature masks for one level.

    The subsample and inner split are drawn ONCE, so every candidate subset is
    judged against identical data - otherwise fitness differences would partly
    measure resampling noise rather than the subsets themselves.
    """
    n_sample = min(n_sample, len(X))
    if n_sample < len(X):
        X, _, y, _ = train_test_split(X, y, train_size=n_sample,
                                      stratify=y, random_state=seed)
    X_fit, X_val, y_fit, y_val = train_test_split(X, y, test_size=0.3,
                                                  stratify=y, random_state=seed)

    def fs_wrapper(position):
        mask = np.asarray(position) > 0.5
        if not mask.any():
            return 1.0                      # empty subset - worst possible
        clf = DecisionTreeClassifier(max_depth=FIT_DEPTH, random_state=seed)
        clf.fit(X_fit[:, mask], y_fit)
        score = f1_score(y_val, clf.predict(X_val[:, mask]), average=average)
        return ALPHA * (1 - score) + BETA * (mask.sum() / len(position))

    return fs_wrapper


def holdout_f1(level, mask):
    """Honest check: fit the subset on the FULL training set, score the real test set.

    Reported as a diagnostic only - never used to choose between subsets, which
    would leak the test set back into selection.
    """
    L = LEVELS[level]
    clf = DecisionTreeClassifier(max_depth=FIT_DEPTH, random_state=RANDOM_STATE)
    clf.fit(L['X_train'][:, mask], L['y_train'])
    return f1_score(L['y_test'], clf.predict(L['X_test'][:, mask]), average=L['average'])


print('  fitness = {:.2f}*(1 - F1) + {:.2f}*(k/{})'.format(ALPHA, BETA, DIM))

---
## 2.4 Running the search

`POP_SIZE x N_ITERS` is the whole compute budget - 20 agents over 50 generations is
1,000 evaluations per search, 12,000 across the run. That is small by metaheuristic
standards and generous for a 40-dimensional binary problem; the convergence curves in
2.5 are the check on whether it was enough. A curve still falling at iteration 50 means
the budget was the binding constraint, not the algorithm.

Every optimizer starts from the same seed, so differences between them are differences
in search strategy rather than luck of the initial population. EvoloPy's optimizers
print per-iteration progress - 600 lines of noise across 12 runs - so stdout is
suppressed and one summary line is printed per search instead.

In [ ]:
import io, time, random, contextlib

POP_SIZE, N_ITERS = 20, 50

runs, rows = {}, []

for level, L in LEVELS.items():
    objf = make_fitness(L['X_train'], L['y_train'], L['average'])
    print(f'  === {level} ({L["average"]} F1) ===')

    for name, optimizer in OPTIMIZERS.items():
        random.seed(RANDOM_STATE)
        np.random.seed(RANDOM_STATE)

        t0 = time.time()
        with contextlib.redirect_stdout(io.StringIO()):
            sol = optimizer(objf, [0.0] * DIM, [1.0] * DIM, DIM, POP_SIZE, N_ITERS)
        elapsed = time.time() - t0

        mask = np.asarray(sol.bestIndividual) > 0.5
        if not mask.any():                          # the guard from 2.3, at the exit
            mask = np.zeros(DIM, bool)
            mask[int(np.argmax(sol.bestIndividual))] = True

        test_f1 = holdout_f1(level, mask)
        runs[(level, name)] = dict(mask=mask, convergence=list(sol.convergence))
        rows.append(dict(level=level, optimizer=name, fitness=float(sol.best_score),
                         n_features=int(mask.sum()), test_f1=test_f1, seconds=elapsed))

        print(f'    {name:<4} fitness={sol.best_score:.5f}  '
              f'k={mask.sum():>2}/{DIM}  test_f1={test_f1:.4f}  {elapsed:5.1f}s')
    print()

fs_results = pd.DataFrame(rows)
print(f'  {len(fs_results)} searches, {fs_results["seconds"].sum() / 60:.1f} min total')

---
## 2.5 Results

Three views, in the order they should be read:

1. **The table** - what each search found, ranked by fitness.
2. **Convergence** - whether the searches actually converged, or ran out of budget.
3. **Feature frequency** - which features six independent searches agreed on.

The baseline row matters most in the table: a subset is only worth having if it holds
its F1 against *all 40 features*. Cutting the feature count is the goal, but not at the
cost of detection.

In [ ]:
fs_results = fs_results.sort_values(['level', 'fitness']).reset_index(drop=True)

for level in LEVELS:
    baseline = holdout_f1(level, np.ones(DIM, bool))
    sub = fs_results[fs_results.level == level]
    print(f'  === {level} ===')
    print(f'    {"rank":<5}{"optimizer":<11}{"fitness":>9}{"k":>5}{"test F1":>10}{"vs all-40":>11}')
    for i, r in enumerate(sub.itertuples(), 1):
        print(f'    {i:<5}{r.optimizer:<11}{r.fitness:>9.5f}{r.n_features:>5}'
              f'{r.test_f1:>10.4f}{r.test_f1 - baseline:>+11.4f}')
    print(f'    {"-":<5}{"all 40":<11}{"-":>9}{DIM:>5}{baseline:>10.4f}{0.0:>+11.4f}')
    print()

### Reading the gap between search F1 and test F1

Read the two score columns together and the gap is the story. A search fitness of 0.008
means validation F1 near 0.99. The same subset scores roughly 0.78 (Level 1) and 0.53
(Level 2) on KDDTest+. That is not a broken search - it is what NSL-KDD is built to do:

- **17 of the 38 attack types in KDDTest+ never appear in KDDTrain+** - 3,750 rows, or
  16.6% of the test set and **29.2% of its attack rows**. R2L and U2R take the brunt
  (7 and 4 novel types), which is why Level 2's macro F1 falls furthest.
- The inner validation split is drawn from the training distribution, so 0.99 there
  measures how cleanly a subset separates *known* attacks. The test set additionally
  asks whether it generalises to attacks nobody trained on.

The consequence for this table: **search fitness does not rank the subsets the way test
F1 does.** Spearman between the two columns is -0.43 on Level 1 and +0.09 on Level 2;
with six searches per level neither is distinguishable from noise, and that is the
finding - there is no evidence here that a better search score buys a better test score.
Level 2's GWO subset is the sharp case: fifth of six by fitness, best of six on test,
and the only subset on either level that beats its all-40 baseline.

The response is *not* to start ranking on the test column. That moves the overfitting up
a level, from the model to the selection procedure, and the test set stops being a test.
It is to carry the gap into Step 3 as a known property of the data - and to judge the
promoted subsets there on a cascade evaluation against KDDTest+, not on the search
score that got them promoted.

### Convergence

Fitness of the best agent at each iteration. A curve that flattens early has converged -
the remaining budget bought nothing. A curve still falling at iteration 50 means the
search was cut short and would reward a larger budget.

Six series converging on nearly the same value is the case where end-of-line labels stop
working - they detach from their lines and read as noise - so identity is carried by the
legend, with the table above as the value-accurate view.

In [ ]:
# Validated categorical palette (light mode) - slots assigned in fixed order,
# one per optimizer, never cycled.
SERIES = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4', '#008300']
SURFACE, INK, INK_2, GRID = '#fcfcfb', '#0b0b0b', '#52514e', '#e3e2df'
OPT_COLOR = dict(zip(OPTIMIZERS, SERIES))


def style_axes(ax):
    ax.set_facecolor(SURFACE)
    ax.grid(True, color=GRID, linewidth=1, linestyle='-')
    ax.set_axisbelow(True)
    for side in ('top', 'right'):
        ax.spines[side].set_visible(False)
    for side in ('left', 'bottom'):
        ax.spines[side].set_color(GRID)
        ax.spines[side].set_linewidth(1)
    ax.tick_params(colors=INK_2, labelsize=9, length=0)


fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), facecolor=SURFACE)

for ax, (level, L) in zip(axes, LEVELS.items()):
    for name in OPTIMIZERS:
        curve = runs[(level, name)]['convergence']
        ax.plot(range(1, len(curve) + 1), curve, color=OPT_COLOR[name], linewidth=2,
                solid_joinstyle='round', solid_capstyle='round', label=name)
    style_axes(ax)
    ax.set_title(f'{level} - {L["average"]} F1', color=INK, fontsize=11,
                 fontweight='semibold', loc='left', pad=10)
    ax.set_xlabel('iteration', color=INK_2, fontsize=9)
    ax.set_ylabel('fitness (lower is better)', color=INK_2, fontsize=9)

axes[0].legend(frameon=False, fontsize=9, labelcolor=INK_2, ncol=2, loc='upper right')
fig.suptitle('Feature-selection search convergence', color=INK, fontsize=13,
             fontweight='semibold', x=0.005, ha='left', y=1.0)
fig.tight_layout()
fig.savefig(Path(RESULTS_DIR) / 'fs_convergence.png', dpi=150,
            facecolor=SURFACE, bbox_inches='tight')
plt.show()

### Feature frequency

How many of the six searches kept each feature, per level. Six independent optimizers
landing on the same feature is a far stronger signal than any single search's subset -
that feature carries information the others cannot substitute for. Features at zero were
discarded by every search and are candidates to drop from collection entirely, which is
what makes the edge tier cheap in Part 2.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 9), facecolor=SURFACE)

for ax, level in zip(axes, LEVELS):
    freq = pd.Series(
        np.sum([runs[(level, n)]['mask'] for n in OPTIMIZERS], axis=0),
        index=FEATURES,
    ).sort_values()

    ax.barh(range(len(freq)), freq.values, height=0.62, color=SERIES[0])
    ax.set_yticks(range(len(freq)))
    ax.set_yticklabels(freq.index, fontsize=8, color=INK_2)
    ax.set_xlim(0, len(OPTIMIZERS) + 0.6)
    ax.set_xticks(range(len(OPTIMIZERS) + 1))
    style_axes(ax)
    ax.grid(axis='y', visible=False)

    # value at the tip - one label per bar, and there is no other value channel here
    for i, v in enumerate(freq.values):
        ax.text(v + 0.12, i, str(int(v)), va='center', fontsize=8, color=INK_2)

    ax.set_title(level, color=INK, fontsize=11, fontweight='semibold', loc='left', pad=10)
    ax.set_xlabel(f'searches keeping the feature (of {len(OPTIMIZERS)})',
                  color=INK_2, fontsize=9)

fig.suptitle('Feature selection frequency across the six searches', color=INK,
             fontsize=13, fontweight='semibold', x=0.005, ha='left', y=1.0)
fig.tight_layout()
fig.savefig(Path(RESULTS_DIR) / 'fs_feature_frequency.png', dpi=150,
            facecolor=SURFACE, bbox_inches='tight')
plt.show()

In [ ]:
# Consensus and rejects - the two ends of the frequency plot, as text
for level in LEVELS:
    freq = pd.Series(np.sum([runs[(level, n)]['mask'] for n in OPTIMIZERS], axis=0),
                     index=FEATURES)
    unanimous = list(freq[freq == len(OPTIMIZERS)].index)
    rejected = list(freq[freq == 0].index)
    print(f'  === {level} ===')
    print(f'    kept by all {len(OPTIMIZERS)} ({len(unanimous):>2}): {", ".join(unanimous) or "-"}')
    print(f'    kept by none     ({len(rejected):>2}): {", ".join(rejected) or "-"}')
    print()

---
## 2.6 Choosing the subsets to carry into Step 3

The plan calls for 3 subsets per level. They are taken as the **top 3 by search
fitness** - the training-derived objective - and *not* by `test_f1`. Ranking on the test
column would be selection on the test set by the back door: the subsets would look
excellent in the Step 3 table and mean nothing, which is the failure the whole
train-only discipline in Step 1 and 2.3 exists to prevent. `test_f1` stays in the saved
record as a diagnostic to read *after* the fact.

Saved to `data/processed/feature_subsets.pkl`. All 12 searches are kept, not just the
promoted 6. Given what the previous section found - fitness rank and test rank barely
relate - the six that did not make the cut are not discards but the control group, and
Step 3 should score them alongside the promoted set before anything is called best.

In [ ]:
N_PROMOTED = 3

promoted, subsets = {}, {}
for level in LEVELS:
    ranked = fs_results[fs_results.level == level].nsmallest(N_PROMOTED, 'fitness')
    promoted[level] = list(ranked.optimizer)
    for name in promoted[level]:
        mask = runs[(level, name)]['mask']
        subsets[(level, name)] = [f for f, keep in zip(FEATURES, mask) if keep]

    print(f'  === {level} - promoting {", ".join(promoted[level])} ===')
    for name in promoted[level]:
        feats = subsets[(level, name)]
        print(f'    {name:<4} ({len(feats):>2} features): {", ".join(feats)}')
    print()

joblib.dump(
    dict(runs=runs, results=fs_results, promoted=promoted, subsets=subsets,
         features=FEATURES,
         config=dict(pop_size=POP_SIZE, n_iters=N_ITERS, alpha=ALPHA, beta=BETA,
                     fit_sample=FIT_SAMPLE, fit_depth=FIT_DEPTH,
                     optimizers=list(OPTIMIZERS), seed=RANDOM_STATE)),
    PROCESSED_DIR / 'feature_subsets.pkl',
)
fs_results.to_csv(Path(RESULTS_DIR) / 'feature_selection_results.csv', index=False)

print(f'  saved {PROCESSED_DIR / "feature_subsets.pkl"}')
print(f'  saved {Path(RESULTS_DIR) / "feature_selection_results.csv"}')

---
## Step 2 complete

| Artifact | Contents |
|---|---|
| `data/processed/feature_subsets.pkl` | all 12 searches + the 6 promoted subsets |
| `results/feature_selection_results.csv` | the ranking table |
| `results/fs_convergence.png` | convergence curves |
| `results/fs_feature_frequency.png` | per-feature agreement across searches |

**Carry forward into Step 3:** the execution matrix is *6 feature sets x 6 algorithms*
per level. Level 1 is the latency-critical tier - it sees every packet, so its metric is
recall (a missed attack never reaches Level 2) and inference time, not accuracy. Level 2
is the precision tier and only ever sees what Level 1 flagged, which means its Step 3
evaluation should run on Level 1's *predicted* attacks rather than on ground-truth
attack rows. Evaluating the two levels independently would report a cascade accuracy the
deployed system never achieves.